In [211]:
import pandas as pd

In [212]:
df1 = pd.read_csv('main_result.csv')
df2 = pd.read_csv('llmeval_result.csv')
df = pd.concat([df1, df2], axis=0)
df = df[df['gpt_model'] == 'gpt-4o']
df = df[df['pe'].isin(['tot', 'got'])]
df = df[df['n_self_alignment'] == 0]
df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
48,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.466667,26.000002,2.700000,0.266667,0.0,0.033333,0.133333,1.000000,0.011111,NaN
49,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.433333,26.142859,2.533334,0.400000,0.0,0.066667,0.200000,0.933333,0.022222,NaN
50,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.533333,26.071430,2.433333,0.466667,0.0,0.100000,0.233333,0.933333,0.033333,NaN
51,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.200000,26.400000,2.466667,0.533333,0.0,0.000000,0.266667,0.833333,0.000000,NaN
52,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.100000,26.068966,2.600000,0.400000,0.0,0.000000,0.200000,0.966667,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.000000,0.000000,0.100000
350,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,0.000000,0.000000,0.100000
351,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,0.000000,0.000000,0.100000
352,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.033333,0.000000,0.000000,0.000000


In [213]:
# if evaluator == 'llm' and target_character in [5, 6], and seed > 5 then drop
df = df[~((df['evaluator'] == 'llm') & (df['target_character'].isin([5, 6])) & (df['seed'] > 4))]

In [214]:
df.groupby(['pe', 'target_character', 'evaluator']).agg({'Evaluation/llm_iteration': ['count']})

Evaluation/llm_iteration
                                                  count
pe  target_character evaluator                         
got 1                hr                              30
                     llm                             30
    2                hr                              30
                     llm                             30
    5                hr                              30
                     llm                             30
    6                hr                              30
                     llm                             30
tot 1                hr                              30
                     llm                             30
    2                hr                              30
                     llm                             30
    5                hr                              30
                     llm                             30
    6                hr                              30
                     llm                             30

In [215]:
# min-max normalization with 'score' column for target_character-wise
def min_max_normalize(df):
    df = df.copy()
    def normalize_group(sub_df):
        min_val = sub_df['score'].min()
        max_val = sub_df['score'].max()
        if max_val > min_val:
            sub_df['score'] = (sub_df['score'] - min_val) / (max_val - min_val)
        else:
            sub_df['score'] = 0.0  # 모든 값이 동일한 경우
        return sub_df
    return df.groupby('target_character').apply(normalize_group).reset_index(drop=True)

normalized = min_max_normalize(df)
normalized

/var/folders/x_/2lt9k5kn52q43m1_z0kp7tfm0000gn/T/ipykernel_90010/3838729616.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('target_character').apply(normalize_group).reset_index(drop=True)


,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
0,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.466667,26.000002,2.700000,0.266667,0.0,0.033333,0.133333,1.000000,0.011111,NaN
1,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.433333,26.142859,2.533334,0.400000,0.0,0.066667,0.200000,0.933333,0.022222,NaN
2,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.533333,26.071430,2.433333,0.466667,0.0,0.100000,0.233333,0.933333,0.033333,NaN
3,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.200000,26.400000,2.466667,0.533333,0.0,0.000000,0.266667,0.833333,0.000000,NaN
4,al39weot,finished,1,got,gpt-4o,2,def,hr,9,0,...,0.100000,26.068966,2.600000,0.400000,0.0,0.000000,0.200000,0.966667,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,y98ybpob,finished,6,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
476,y98ybpob,finished,6,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,0.000000,0.000000,0.033333
477,y98ybpob,finished,6,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,0.000000,0.000000,0.033333
478,y98ybpob,finished,6,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.166667,0.000000,0.000000,0.000000


In [216]:
avg_df = normalized.groupby(['gpt_model', 'pe', 'Evaluation/llm_iteration', 'evaluator', 'seed']).agg({'score': 'mean'}).reset_index()
avg_df['target_character'] = 'mean'
avg_df

,gpt_model,pe,Evaluation/llm_iteration,evaluator,seed,score,target_character
0,gpt-4o,got,1,hr,1,0.000000,mean
1,gpt-4o,got,1,hr,2,0.000000,mean
2,gpt-4o,got,1,hr,3,0.002778,mean
3,gpt-4o,got,1,hr,4,0.003704,mean
4,gpt-4o,got,1,hr,5,0.051389,mean
...,...,...,...,...,...,...,...
175,gpt-4o,tot,6,llm,1,0.000000,mean
176,gpt-4o,tot,6,llm,2,0.171296,mean
177,gpt-4o,tot,6,llm,3,0.039583,mean
178,gpt-4o,tot,6,llm,4,0.043333,mean


In [217]:
df = pd.concat([avg_df], ignore_index=True)

In [218]:
def make_table(df: pd.DataFrame, ref_label: str = "PCGRLLM") -> pd.DataFrame:
    def agg_stats(sub_df, label):
        grouped = (
            sub_df.groupby(['pe', 'evaluator'])
            .agg({'score': ['mean']})
            .reset_index()
        )
        grouped.columns = ['pe', 'evaluator', f'{label} (mean)']
        return grouped

    # 각 mode 정의
    zero_mask  = (df['Evaluation/llm_iteration'] == 1)
    fb_mask    = (df['Evaluation/llm_iteration'] == 6)

    base_df  = agg_stats(df[zero_mask], "ZS")
    fb_df    = agg_stats(df[fb_mask],   "+FB")

    pk = ['pe', 'evaluator']
    merged = (
        base_df.merge(fb_df, on=pk, how='outer')
    )

    return merged

def make_perf_table_by_targets(df, targets=(1, 2, 5, 6, 'mean')):
    tables = []
    for t in targets:
        
        sub = df[df['target_character'] == t]
        
        if sub.empty:
            continue
        tbl = make_table(sub)           # 이미 mean/std 포함 버전
        tbl.insert(0, 'target_character', t)
        tables.append(tbl)
    if not tables:
        return pd.DataFrame()
    return pd.concat(tables, ignore_index=True)


In [229]:
perf_table_gpt = make_perf_table_by_targets(df)
# add delta columns, by subtracting +FB - ZS
perf_table_gpt['Delta (mean)'] = perf_table_gpt['+FB (mean)'] - perf_table_gpt['ZS (mean)']
# drop target_character column
perf_table_gpt = perf_table_gpt.drop(columns=['target_character'])
# sort by pe, [tot, got], evaluator [human, llm]
perf_table_gpt = perf_table_gpt.sort_values(by=['pe', 'evaluator'], key=lambda x: x.map({'tot': 0, 'got': 1, 'hr': 0, 'llm': 1})).reset_index(drop=True)
perf_table_gpt

,pe,evaluator,ZS (mean),+FB (mean),Delta (mean)
0,tot,hr,0.000823,0.162860,0.162037
1,tot,llm,0.261204,0.049776,-0.211427
2,got,hr,0.052726,0.142335,0.089609
3,got,llm,0.125000,0.068750,-0.056250
